# AITR Lab02 — Run everything (Google Colab ready)

**Topic:** Context-Window Degradation / evidence position × citation support  
**Student:** Madhukar Kumar · MHM2026001 · IIIT Allahabad

### How to use on Colab
1. Upload this `.ipynb` to Google Colab (or open from GitHub).
2. Run **Runtime → Run all**.
3. Default mode is **offline** (no API key): it clones the project and rebuilds tables/figures from the locked Groq run.

If the GitHub repo is **private**, paste a Personal Access Token in the setup cell (`GH_TOKEN`), or upload a project zip instead.

## 0. Flags (edit here)

In [ ]:
# === edit these if needed ===
GITHUB_URL = "https://github.com/mkumariiita/AITR_Lab02_MHM2026001_ContextWindowDegradation.git"
REPO_DIR_NAME = "AITR_Lab02_MHM2026001_ContextWindowDegradation"

# If the repo is private, paste a GitHub PAT here (repo read access). Leave empty if public.
GH_TOKEN = ""  # e.g. "ghp_xxxxxxxx" — never commit a real token

# If clone fails, set True and the next cell will ask you to upload a .zip of the project folder
FORCE_ZIP_UPLOAD = False

# Live Groq calls (costs free-tier quota). Keep False for paper reproduction.
RUN_LIVE_API = False
LIVE_LIMIT = 1
LIVE_SLEEP = 2.5

print("flags ready")

## 1. Install packages

In [ ]:
import sys
import subprocess

pkgs = [
    "PyYAML>=6.0.1",
    "requests>=2.31.0",
    "python-dotenv>=1.0.0",
    "matplotlib>=3.8.0",
    "pandas>=2.0.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("packages installed:", ", ".join(pkgs))

## 2. Get the project (clone GitHub, or upload zip)

In [ ]:
from pathlib import Path
import os
import shutil
import zipfile

IN_COLAB = Path("/content").exists()
WORK = Path("/content" if IN_COLAB else ".").resolve()
ROOT = WORK / REPO_DIR_NAME

def project_looks_ok(root: Path) -> bool:
    return (
        (root / "src" / "position_citation").is_dir()
        and (root / "data" / "packets").is_dir()
        and (root / "results" / "tables" / "results_run_20261030T142328Z.csv").is_file()
    )

def clone_repo(dest: Path) -> None:
    if dest.exists():
        shutil.rmtree(dest)
    url = GITHUB_URL
    if GH_TOKEN.strip():
        # https://github.com/owner/repo.git → https://<token>@github.com/owner/repo.git
        url = GITHUB_URL.replace("https://", f"https://{GH_TOKEN.strip()}@")
    cmd = ["git", "clone", "--depth", "1", url, str(dest)]
    print("Running:", "git clone --depth 1 <url> " + str(dest))
    subprocess.check_call(cmd)

def upload_zip_and_extract(dest: Path) -> None:
    if not IN_COLAB:
        raise RuntimeError("Zip upload helper is for Google Colab. Unzip the project locally instead.")
    from google.colab import files
    print("Upload a .zip of the project folder (must contain data/, src/, results/, scripts/)...")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded")
    zname = next(iter(uploaded))
    zpath = WORK / zname
    if dest.exists():
        shutil.rmtree(dest)
    extract_dir = WORK / "_unzipped"
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(extract_dir)
    # find the folder that contains src/position_citation
    candidates = [p for p in extract_dir.rglob("position_citation") if p.is_dir()]
    if not candidates:
        raise RuntimeError("Could not find src/position_citation inside the zip")
    project_root = candidates[0].parent.parent  # .../src/position_citation → project root
    shutil.move(str(project_root), str(dest))
    print("Extracted project to", dest)

# Local case: notebook already inside the repo
here = Path.cwd().resolve()
if (here.name == "notebooks" and project_looks_ok(here.parent)):
    ROOT = here.parent
    print("Using local project at", ROOT)
elif project_looks_ok(here):
    ROOT = here
    print("Using local project at", ROOT)
elif FORCE_ZIP_UPLOAD:
    upload_zip_and_extract(ROOT)
else:
    try:
        clone_repo(ROOT)
    except Exception as exc:
        print("Clone failed:", exc)
        print("Falling back to zip upload...")
        upload_zip_and_extract(ROOT)

assert project_looks_ok(ROOT), f"Project incomplete at {ROOT}"
os.chdir(ROOT)
print("ROOT =", ROOT)
print("cwd  =", Path.cwd())

## 3. Paths + imports

In [ ]:
import json
import platform
import csv

import yaml
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

PRIMARY_CSV = ROOT / "results" / "tables" / "results_run_20261030T142328Z.csv"
PRIMARY_RAW = ROOT / "results" / "raw" / "run_20261030T142328Z"
CONFIG_PATH = ROOT / "configs" / "experiment.yaml"

print("python:", sys.version.split()[0], "|", platform.platform())
print("IN_COLAB:", IN_COLAB)
print("PRIMARY_CSV:", PRIMARY_CSV.is_file(), PRIMARY_CSV)
print("PRIMARY_RAW files:", len(list(PRIMARY_RAW.glob('item_*.json'))))
print("packets:", len(list((ROOT / 'data' / 'packets').glob('item_*.json'))))

## 4. Environment + packet smoke checks

In [ ]:
from position_citation.packets import load_all_packets, load_packet
from position_citation.assemble import assemble_ordered_passages, build_prompt
from position_citation.score import score_response

packets = load_all_packets(ROOT / "data" / "packets")
assert len(packets) == 12, len(packets)

pkt = load_packet(ROOT / "data" / "packets" / "item_01.json")
begin_ids = [p["id"] for p in assemble_ordered_passages(pkt, "begin")]
middle_ids = [p["id"] for p in assemble_ordered_passages(pkt, "middle")]
end_ids = [p["id"] for p in assemble_ordered_passages(pkt, "end")]

assert begin_ids[:2] == ["P2", "P5"], begin_ids
assert end_ids[-2:] == ["P2", "P5"], end_ids
assert "P2" in middle_ids and "P5" in middle_ids

good = score_response(
    pkt,
    "Low-prior-knowledge gains in Study A [P2]; Study B finds no average effect [P5].",
)
assert good["answer_correctness"] == 1
assert good["evidence_coverage"] == 1.0
assert good["citation_support"] == 1.0

cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
print("packets:", len(packets))
print("model freeze:", cfg["model"]["name"], "|", cfg["model"]["provider"])
print("positions:", cfg["positions"])
print("item_01 order begin / middle / end:")
print(" ", begin_ids)
print(" ", middle_ids)
print(" ", end_ids)
print("smoke: PASS")

## 5. Example prompts (begin / middle / end)

Same gold passages; only order changes.

In [ ]:
demo = load_packet(ROOT / "data" / "packets" / "item_02.json")
for pos in ("begin", "middle", "end"):
    prompt = build_prompt(demo, pos, prompt_version=cfg.get("prompt_version", "cite_v1"))
    order = [p["id"] for p in assemble_ordered_passages(demo, pos)]
    print(f"\n=== {pos} | order={order} | words≈{len(prompt.split())} ===")
    lines = [ln for ln in prompt.splitlines() if ln.startswith("[") or ln.startswith("Question:")]
    for ln in lines:
        print(ln[:140])

## 6. Optional live Groq API run (OFF by default)

Needs a key. On Colab, set it in the cell below (do not share the notebook with the key filled in).

In [ ]:
if not RUN_LIVE_API:
    print("Skipping live API (RUN_LIVE_API=False). Using locked primary CSV for paper numbers.")
else:
    # Paste key only when RUN_LIVE_API=True. Prefer Colab Secrets if available.
    GROQ_API_KEY = ""  # e.g. "gsk_..."
    if not GROQ_API_KEY:
        try:
            from google.colab import userdata
            GROQ_API_KEY = userdata.get("GROQ_API_KEY")
        except Exception:
            GROQ_API_KEY = os.environ.get("LLM_API_KEY", "")

    env_path = ROOT / ".env"
    env_path.write_text(
        "\n".join([
            "LLM_BACKEND=openai_compat",
            f"LLM_API_KEY={GROQ_API_KEY}",
            "LLM_BASE_URL=https://api.groq.com/openai/v1",
            "LLM_MODEL=openai/gpt-oss-20b",
            "",
        ]),
        encoding="utf-8",
    )
    assert GROQ_API_KEY, "Set GROQ_API_KEY in this cell or Colab Secrets"

    cmd = [
        sys.executable,
        str(ROOT / "scripts" / "run_baseline.py"),
        "--backend", "openai_compat",
        "--limit", str(LIVE_LIMIT),
        "--sleep", str(LIVE_SLEEP),
    ]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd, cwd=ROOT)
    print("Live run finished. Paper numbers below still come from the locked primary CSV.")

## 7. Evaluate locked results → tables + figures

Main reproduction path for the paper / Gate G3.

In [ ]:
import evaluate_results as ev

rows = ev.load_rows(PRIMARY_CSV)
means = ev.means_by_position(rows)
cases = ev.failure_cases(rows)

out_tables = ROOT / "results" / "tables"
out_fig = ROOT / "results" / "figures"
ev.write_summary_csv(means, out_tables / "g3_summary_by_position.csv")
ev.write_failure_csv(cases, out_tables / "g3_failure_cases.csv")
ev.plot_figures(means, rows, out_fig)
ev.write_meta(rows, means, out_tables / "g3_eval_meta.json", PRIMARY_CSV)

print(f"rows={len(rows)} items={len({r['item_id'] for r in rows})} failure_rows={len(cases)}")
for pos in ("begin", "middle", "end"):
    m = means[pos]
    print(
        f"{pos:6s}  coverage={m['mean_coverage']:.2f}  "
        f"citation={m['mean_citation_support']:.2f}  "
        f"keyword_corr={m['mean_correctness']:.2f}"
    )

## 8. Summary table (paper Table 1)

In [ ]:
import pandas as pd

summary = pd.read_csv(out_tables / "g3_summary_by_position.csv")
display(summary)

## 9. Figures

In [ ]:
for name in (
    "fig_position_means.png",
    "fig_per_item_correctness.png",
    "fig_failure_tags.png",
):
    path = out_fig / name
    display(Markdown(f"### {name}"))
    display(Image(filename=str(path)))

## 10. Deep analysis — keyword misses (coverage/citation still perfect)

In [ ]:
print(f"Flagged rows: {len(cases)}\n")

samples = [
    ("item_02", "end"),
    ("item_07", "begin"),
    ("item_08", "middle"),
    ("item_05", "middle"),
]
for item_id, pos in samples:
    path = PRIMARY_RAW / f"{item_id}_{pos}.json"
    data = json.loads(path.read_text(encoding="utf-8"))
    scores = data["scores"]
    print(f"--- {item_id} / {pos} ---")
    print(
        f"cov={scores['evidence_coverage']} cite={scores['citation_support']} "
        f"corr={scores['answer_correctness']} citations={scores['citations']}"
    )
    print(data["response"][:350].replace("\n", " "))
    print()

## 11. Takeaway

On this short synthetic protocol, **evidence coverage** and **citation support** are **1.00** at begin, middle, and end — a **null / negative result** for the expected middle-position drop. Keyword correctness varies but is a weak secondary metric (paraphrase mismatches).

In [ ]:
display(Markdown(
    "**Done.** Offline reproduction complete.\n\n"
    f"- Summary: `{(out_tables / 'g3_summary_by_position.csv').relative_to(ROOT)}`\n"
    f"- Failures: `{(out_tables / 'g3_failure_cases.csv').relative_to(ROOT)}`\n"
    f"- Figures: `{out_fig.relative_to(ROOT)}/`\n"
    f"- Locked raw run: `{PRIMARY_RAW.relative_to(ROOT)}`\n"
    f"- Repo: `{GITHUB_URL}`"
))